In [ ]:
#Importing necessary libraries
import os
import nibabel as nib
from nilearn.decomposition import CanICA
from nilearn import plotting
import matplotlib.pyplot as plt
from nilearn.maskers import NiftiMapsMasker
import glob
from nilearn.image import iter_img, new_img_like
from nilearn.plotting import plot_stat_map, plot_prob_atlas, show
from nilearn import datasets, image
from scipy.stats import pearsonr, ttest_ind
import numpy as np

###
ad_files = glob.glob("C:/Users/elifn/Desktop/ADNI/*_S_*/*_f_n/swauf*.nii")
cn_files = glob.glob("C:/Users/elifn/Desktop/ADNI/CN/*_S_*/*_f_n/swauf*.nii")

for ad in ad_files:
    img = nib.load(ad)
    boyut = img.shape # [X, Y, Z, time]
    voxel_size = img.header.get_zooms()
    isim = ad.split('\\')[-1] 
    print(f"Dosya: {isim}")
    print(f"Boyut (Dim): {boyut} | Voksel Boyutu: {voxel_size[:3]}")
    print("-" * 50)
print("-------------------------")
for cn in cn_files:
    img = nib.load(cn)
    boyut = img.shape # [X, Y, Z, time]
    voxel_size = img.header.get_zooms()
    isim = cn.split('\\')[-1] 
    print(f"Dosya: {isim}")
    print(f"Boyut (Dim): {boyut} | Voksel Boyutu: {voxel_size[:3]}")
    print("-" * 50)

    ##
all_subjects = ad_files + cn_files
canica = CanICA(n_components=30, memory=None, memory_level=0, threshold=3.0, verbose=1, random_state=42, mask_strategy="whole-brain-template", n_jobs=2)
canica.fit(all_subjects)
components = canica.components_img_
plot_prob_atlas(components, title="All ICA components")

group_components_img = canica.components_img_
print(group_components_img.shape)

fig, axes = plt.subplots(5, 6, figsize=(15, 16))
axes = axes.flatten()

for i, (cur_img, ax) in enumerate(zip(iter_img(components), axes)):
    plot_stat_map(
        cur_img,
        display_mode="z",
        title=f"IC {int(i)}",
        cut_coords=1,
        vmax=0.05,
        vmin=-0.05, 
        colorbar=True, ###It was false but vmax and vmin are the boundaries for it
        axes=ax         
    )

plt.show() ##Resting state ICNs 
####

atlas = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=7, resolution_mm=2) 
atlas_data = image.load_img(atlas.maps).get_fdata()
dmn_mask_3d = np.zeros_like(atlas_data)
ecn_mask_3d = np.zeros_like(atlas_data) 
for i, label in enumerate(atlas.labels):
    label_str = label.decode('utf-8') if isinstance(label, bytes) else str(label)
    roi_value = i + 1
    if 'Default' in label_str:
        dmn_mask_3d[atlas_data == roi_value] = 1
    elif 'Cont' in label_str:
        ecn_mask_3d[atlas_data == roi_value] = 1

dmn_resampled = image.resample_to_img(new_img_like(image.load_img(atlas.maps), dmn_mask_3d), components, interpolation='nearest')
ecn_resampled = image.resample_to_img(new_img_like(image.load_img(atlas.maps), ecn_mask_3d), components, interpolation='nearest')

dmn_1d = dmn_resampled.get_fdata().flatten()
ecn_1d = ecn_resampled.get_fdata().flatten()
ica_data = components.get_fdata()

dmn_scores = []
ecn_scores = []

for i in range(ica_data.shape[3]):
    ic_map = ica_data[..., i].flatten()
    r_dmn, _ = pearsonr(ic_map, dmn_1d)
    r_ecn, _ = pearsonr(ic_map, ecn_1d)
    
    dmn_scores.append(r_dmn)
    ecn_scores.append(r_ecn)

best_dmn = np.argmax(dmn_scores)
best_ecn = np.argmax(ecn_scores)

print(f"DMN -> ICA{best_dmn}")

print(f"ECN -> ICA{best_ecn}")

###computation
dmn_img = image.index_img(components, 5)
ecn_img = image.index_img(components, 21)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

plot_stat_map(dmn_img, axes=axes[0], title="IC_5: Default Mode Network (DMN)", 
              display_mode='z', cut_coords=[16, 26, 36])

plot_stat_map(ecn_img, axes=axes[1], title="IC_21: Yürütücü Kontrol Ağı (ECN)", 
              display_mode='z', cut_coords=[34, 44, 54])

plt.show()

##
ad_folders = set([os.path.dirname(f) for f in ad_files])
cn_folders = set([os.path.dirname(f) for f in cn_files])

# Matris kuracağımız için 'zscore_sample' kalıyor.
masker = NiftiMapsMasker(maps_img=components, standardize='zscore_sample', verbose=0)

ad_time_series = []
for folder in ad_folders:
 
    subject_3d_files = sorted(glob.glob(os.path.join(folder, "swauf*.nii")))
    rp_file = glob.glob(os.path.join(folder, "rp_*.txt"))
    ts = masker.fit_transform(subject_3d_files, confounds=rp_file[0])
    ad_time_series.append(ts)

cn_time_series = []
for folder in cn_folders:
    subject_3d_files = sorted(glob.glob(os.path.join(folder, "swauf*.nii")))
    
    rp_file = glob.glob(os.path.join(folder, "rp_*.txt"))
    
    ts = masker.fit_transform(subject_3d_files, confounds=rp_file[0])
    cn_time_series.append(ts)

print("Dual Regression")
print(f"AD Grubundan {len(ad_time_series)}, CN Grubundan {len(cn_time_series)}")

example_data = ad_time_series[0]
print(f"From sample: {example_data}")

###

from nilearn.connectome import ConnectivityMeasure

correlation_measure = ConnectivityMeasure(kind='correlation')

ad_matrices = correlation_measure.fit_transform(ad_time_series) 
cn_matrices = correlation_measure.fit_transform(cn_time_series) 

ad_mean_matrix = np.mean(ad_matrices, axis=0)
cn_mean_matrix = np.mean(cn_matrices, axis=0)

print(f"AD Grubu Ortalama Matris Boyutu: {ad_mean_matrix.shape}")
print(f"CN Grubu Ortalama Matris Boyutu: {cn_mean_matrix.shape}")

##
import matplotlib.pyplot as plt
from nilearn.plotting import plot_matrix


fig, axes = plt.subplots(1, 2, figsize=(16, 7))


plot_matrix(
    ad_mean_matrix, 
    axes=axes[0], 
    vmax=1.0, vmin=-1.0, # Pearson korelasyonu -1 ile +1 arasındadır
    colorbar=False,      # İki tane renk skalasına gerek yok, bir tane yeter
    title="Alzheimer (AD) Ortalama FC (30x30)"
)

plot_matrix(
    cn_mean_matrix, 
    axes=axes[1], 
    vmax=1.0, vmin=-1.0, 
    colorbar=True,       # Renk skalasını sağdaki grafiğe ekliyoruz
    title="Sağlıklı Kontrol (CN) Ortalama FC (30x30)"
)

plt.show()

ad_dmn_ecn_crosstalk = ad_mean_matrix[5, 21]
cn_dmn_ecn_crosstalk = cn_mean_matrix[5, 21]

print(f"DMN (IC_5) and ECN (IC_21) Cross-talk")
print(f"CN Grubu: {cn_dmn_ecn_crosstalk:.4f}")
print(f"AD Grubu: {ad_dmn_ecn_crosstalk:.4f}")

## I just wanted to try the statistical part for my learning... 
from scipy.stats import ttest_ind

ad_individual_crosstalk = ad_matrices[:, 5, 21]
cn_individual_crosstalk = cn_matrices[:, 5, 21]

t_stat, p_val = ttest_ind(ad_individual_crosstalk, cn_individual_crosstalk, alternative='greater')

print("DMN-ECN Cross-talk:")
print(f"AD: {ad_individual_crosstalk.mean():.4f}")
print(f"CN: {cn_individual_crosstalk.mean():.4f}")
print(f"T-Score: {t_stat:.3f}")
print(f"P-Value: {p_val:.4f}\n")

if p_val < 0.05:
    print("Result: there is some statistical meaning")
else:
    print("Result: sample size can make an impact on the test of course")